# HW1 — From Tokens to Classifiers


In [ ]:
# Uncomment the next line if imports fail, then run this cell again.
# %pip install -q datasets transformers

from datasets import load_dataset
from transformers import AutoTokenizer
import pandas as pd

## 1. Load the first 500 test pairs

`Helsinki-NLP/opus-100` with config `en-tr` is a parallel corpus: each row is the same sentence in English and Turkish.

The field is `translation`, a dict with keys `"en"` and `"tr"`. Take the `test` split and keep only the first 500 rows.

In [ ]:
N_SENTENCES = 500

test_split = load_dataset("Helsinki-NLP/opus-100", "en-tr", split="test")
#print(test_split)  # Let me see what is inside test_split
#print(test_split[32])  # Show the item number n for structure/fields
pairs = test_split.select(range(N_SENTENCES))

translations = pairs["translation"]
en_texts = [row["en"] for row in translations]
tr_texts = [row["tr"] for row in translations]

print(len(en_texts), "pairs")
print("EN:", en_texts[0])
print("TR:", tr_texts[0])

500 pairs
EN: ['Russian leaders reacted to the announcement of US plans to place missile defence bases in eastern Poland.', 'Mr Popular. Time to meet your friends.', "I'll check this bus.", "Yeah, it's been raining, hasn't it?", '- Yeah, like a... you know, like a date.']
TR: ["Rus liderleri Amerika Birleşik Devletleri'nin Polonya'nın doğusuna füze üssü kurma planlarına tepkili.", 'Bay Popüler, arkadaşlarınızla tanışma zamanı.', 'Ben otobüsü bulayım.', 'Evet, yağıyor, değil mi?', '- Öğle yemeğini akşam yiyeceğiz.']


## 2. Load the four tokenizers

| Name | What it was built for |
|---|---|
| `gpt2` | English BPE. Turkish was not a design goal. |
| `bert-base-multilingual-cased` | WordPiece shared across many languages, including Turkish. |
| `dbmdz/bert-base-turkish-cased` | WordPiece trained for Turkish. |
| `Qwen/Qwen2.5-1.5B-Instruct` | A recent multilingual BPE used by a chat model. |

`AutoTokenizer.from_pretrained` reads each model's vocabulary. It does not load the neural network.

In [ ]:
TOKENIZER_NAMES = [
    "gpt2",
    "bert-base-multilingual-cased",
    "dbmdz/bert-base-turkish-cased",
    "Qwen/Qwen2.5-1.5B-Instruct",
]

tokenizers = {
    name: AutoTokenizer.from_pretrained(name)
    for name in TOKENIZER_NAMES
}

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

## 3. One sentence, by hand

Before averaging 500 rows, check one pair so the definition is visible.

- `text.split()` is the whitespace-word count the assignment asks for.
- `encode(..., add_special_tokens=False, truncation=False)` is the token count. `truncation=False` matters because a BERT tokenizer would otherwise be allowed to cut a long sentence at 512 tokens and undercount it. These sentences are short; the flag keeps the measurement honest.
- The last prints show what `[CLS]` and `[SEP]` would have added for multilingual BERT.

In [ ]:
def n_tokens(tokenizer, text: str) -> int:
    ids = tokenizer.encode(text, add_special_tokens=False, truncation=False)
    return len(ids)


sample_en = en_texts[0]
sample_tr = tr_texts[0]
sample_tok = tokenizers["gpt2"]

print("EN words:", sample_en.split())
print("EN gpt2 pieces:", sample_tok.tokenize(sample_en))
print(
    "EN fertility:",
    n_tokens(sample_tok, sample_en) / len(sample_en.split()),
)
print()
print("TR words:", sample_tr.split())
print("TR gpt2 pieces:", sample_tok.tokenize(sample_tr))
print(
    "TR fertility:",
    n_tokens(sample_tok, sample_tr) / len(sample_tr.split()),
)

bert = tokenizers["bert-base-multilingual-cased"]
print()
print("mBERT tokens, special tokens off:", n_tokens(bert, sample_en))
print(
    "mBERT tokens, special tokens on:",
    len(bert.encode(sample_en, add_special_tokens=True, truncation=False)),
)
print(
    "those extra pieces:",
    bert.convert_ids_to_tokens(bert.encode(sample_en, add_special_tokens=True)),
)

EN words: ['Russian', 'leaders', 'reacted', 'to', 'the', 'announcement', 'of', 'US', 'plans', 'to', 'place', 'missile', 'defence', 'bases', 'in', 'eastern', 'Poland.']
EN gpt2 pieces: ['Russian', 'Ġleaders', 'Ġreacted', 'Ġto', 'Ġthe', 'Ġannouncement', 'Ġof', 'ĠUS', 'Ġplans', 'Ġto', 'Ġplace', 'Ġmissile', 'Ġdefence', 'Ġbases', 'Ġin', 'Ġeastern', 'ĠPoland', '.']
EN fertility: 1.0588235294117647

TR words: ['Rus', 'liderleri', 'Amerika', 'Birleşik', "Devletleri'nin", "Polonya'nın", 'doğusuna', 'füze', 'üssü', 'kurma', 'planlarına', 'tepkili.']
TR gpt2 pieces: ['Rus', 'Ġl', 'ider', 'ler', 'i', 'ĠAmer', 'ika', 'ĠBir', 'le', 'ÅŁ', 'ik', 'ĠDev', 'let', 'ler', 'i', "'", 'nin', 'ĠPol', 'ony', 'a', "'", 'n', 'Ä±', 'n', 'Ġdo', 'ÄŁ', 'us', 'una', 'Ġf', 'Ã¼', 'ze', 'ĠÃ', '¼', 'ss', 'Ã¼', 'Ġk', 'ur', 'ma', 'Ġplan', 'lar', 'Ä±', 'na', 'Ġt', 'ep', 'k', 'ili', '.']
TR fertility: 3.9166666666666665

mBERT tokens, special tokens off: 20
mBERT tokens, special tokens on: 22
those extra pieces: ['[CLS]', 'Ru

## 4. Fertility and the Turkish/English ratio

For every tokenizer, do this separately on the English list and the Turkish list:

- fertility of one sentence = its token count divided by its whitespace-word count
- fertility of a language = the average of those sentence fertilities
- total tokens = the sum of token counts (this is the numerator and denominator of the ratio)

Skip a sentence only if `split()` is empty, so you never divide by zero. The totals still include every sentence.

The dict printed at the end is the `fertility` block you will later paste into `results.json`. Do not type the numbers in by hand. Read them from this output when the rest of the homework is done.

In [ ]:
def fertility_and_total(tokenizer, texts):
    per_sentence = []
    total_tokens = 0
    for text in texts:
        n_words = len(text.split())
        n_tok = n_tokens(tokenizer, text)
        total_tokens += n_tok
        if n_words == 0:
            continue
        per_sentence.append(n_tok / n_words)
    mean_fertility = sum(per_sentence) / len(per_sentence)
    return mean_fertility, total_tokens


fertility = {}
rows = []
for name in TOKENIZER_NAMES:
    tok = tokenizers[name]
    fert_en, total_en = fertility_and_total(tok, en_texts)
    fert_tr, total_tr = fertility_and_total(tok, tr_texts)
    fertility[name] = {"en": fert_en, "tr": fert_tr}
    rows.append(
        {
            "tokenizer": name,
            "fertility_en": fert_en,
            "fertility_tr": fert_tr,
            "total_tokens_en": total_en,
            "total_tokens_tr": total_tr,
            "tr_over_en": total_tr / total_en,
        }
    )

summary = pd.DataFrame(rows)
summary

In [ ]:
fertility

## 5. Three Turkish words with long suffix chains

Turkish often packs a whole phrase into one word:

- `evlerimizden` — ev-ler-imiz-den, roughly "from our houses"
- `gelemeyeceklermiş` — a verb plus several suffixes, roughly "apparently they will not be able to come"
- `Afyonkarahisarlılaştıramadıklarımızdan` — a city name with a long stack of suffixes

`tokenize` shows the pieces. More pieces means that word is more expensive for that tokenizer.

GPT-2 marks a leading space with `Ġ`. A word in the middle of a sentence is usually stored with that space. The assignment asks for the words themselves, so the table uses the bare word. The extra print shows the same word with a leading space, which is what GPT-2 would see inside a sentence.

In [ ]:
TURKISH_WORDS = [
    "evlerimizden",
    "gelemeyeceklermiş",
    "Afyonkarahisarlılaştıramadıklarımızdan",
]

for word in TURKISH_WORDS:
    print(word)
    for name in TOKENIZER_NAMES:
        pieces = tokenizers[name].tokenize(word)
        print(f"  {name}: {len(pieces)} -> {pieces}")
    print()

print("Same first word with a leading space (how BPE often sees it in a sentence):")
spaced = " " + TURKISH_WORDS[0]
for name in TOKENIZER_NAMES:
    pieces = tokenizers[name].tokenize(spaced)
    print(f"  {name}: {len(pieces)} -> {pieces}")

## 6. Which tokenizer is most expensive for Turkish?

GPT-2 is the most expensive tokenizer for Turkish. Its vocabulary was built for English, so a Turkish word with a long suffix chain is not one piece. It is cut into many small tokens, including raw bytes for letters such as ş, ı, and ü. On the first sentence pair this is already about 1.06 tokens per English word and about 3.92 per Turkish word, while the Turkish-specific tokenizer keeps those chunks together. The same Turkish text therefore fills the context window sooner, so less of it fits. It also costs more on an API, because the bill is per token.

# Part B — Inside the generator

The model is `Qwen/Qwen2.5-1.5B-Instruct` in half precision (`float16`). Run this part on a GPU (Colab T4). Part A did not load the network. This part does.

Three tasks:

1. Given `The capital of Turkey is`, print the 10 tokens the model thinks are most likely to come next, with their probabilities.
2. Generate 5 continuations of one prompt three ways: greedy, temperature 0.7, and temperature 1.5. `top_p=1` and a fixed seed.
3. Time 200 new tokens with the key/value cache on and off. Three runs each. Report the mean.

## 1. Load the model in half precision

Half precision stores each weight in 16 bits instead of 32, which is what a T4 GPU expects for this model. The tokenizer from Part A is only the vocabulary. `AutoModelForCausalLM` is the network that produces next-token scores.

In [6]:
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
SEED = 42

qwen_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
qwen = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

print("precision as loaded:", qwen.dtype)

qwen = qwen.half()

print("precision after half:", qwen.dtype)

qwen.to("cuda")
qwen.eval()

print("device:", next(qwen.parameters()).device)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

precision as loaded: torch.bfloat16
precision after half: torch.float16
device: cuda:0


## 2. Top 10 next tokens

The model reads the prompt and produces one score per vocabulary entry for the next position. Softmax turns those scores into probabilities that sum to 1. We take the 10 largest.

The prompt is used as written. It is not wrapped in the chat template, because the assignment asks for this exact string.

In [10]:
CAPITAL_PROMPT = "The capital of Turkey is"

capital_inputs = qwen_tokenizer(CAPITAL_PROMPT, return_tensors="pt").to(qwen.device)
#print(capital_inputs)
with torch.no_grad():
    capital_logits = qwen(**capital_inputs).logits[0, -1]
    print(capital_logits)
capital_probs = torch.softmax(capital_logits.float(), dim=-1)
top_probs, top_ids = torch.topk(capital_probs, 10)

for rank, (prob, token_id) in enumerate(zip(top_probs, top_ids), start=1):
    token_id = token_id.item()
    piece = qwen_tokenizer.convert_ids_to_tokens(token_id)
    text = qwen_tokenizer.decode([token_id])
    print(f"{rank:2d}  p={prob.item():.6f}  text={text!r}  token={piece!r}")

tensor([ 9.9062,  7.5977,  6.1719,  ..., -1.0322, -1.0322, -1.0322],
       device='cuda:0', dtype=torch.float16)
 1  p=0.199948  text=' Ankara'  token='ĠAnkara'
 2  p=0.087351  text=' Istanbul'  token='ĠIstanbul'
 3  p=0.072416  text=' located'  token='Ġlocated'
 4  p=0.071294  text=' __'  token='Ġ__'
 5  p=0.039372  text=' ______'  token='Ġ______'
 6  p=0.033677  text=' ____'  token='Ġ____'
 7  p=0.026228  text='____'  token='____'
 8  p=0.025821  text=' in'  token='Ġin'
 9  p=0.025821  text=' a'  token='Ġa'
10  p=0.025421  text=' ('  token='Ġ('


## 3. Five continuations, three decoding settings

The prompt is fixed: `The student opened the old notebook and`. Change that string if you want a different topic, then keep it fixed for all 15 generations.

- **Greedy:** always pick the single most likely next token. The seed does not matter. Five runs should match.
- **Temperature 0.7:** sample at random, but lower temperature makes likely tokens even more likely. Output varies, and it usually stays on topic.
- **Temperature 1.5:** the same sampling, with the distribution flattened. Unlikely tokens show up more often, so the five continuations should drift and disagree.

`top_p=1` means nucleus sampling is off: every token stays eligible. The seed is set once before each group of 5, then left alone. Resetting it before every sample would make those 5 copies identical. Each new token is capped at 40 so the comparison stays readable.

In [ ]:
CONTINUATION_PROMPT = "The weather was nice and sunny on sunday"
N_CONTINUATIONS = 5
MAX_NEW_TOKENS = 20

def five_continuations(do_sample, temperature=None):
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    encoded = qwen_tokenizer(CONTINUATION_PROMPT, return_tensors="pt").to(qwen.device)
    prompt_length = encoded["input_ids"].shape[1]
    texts = []
    for _ in range(N_CONTINUATIONS):
        generate_kwargs = {
            "max_new_tokens": MAX_NEW_TOKENS,
            "pad_token_id": qwen_tokenizer.eos_token_id,
            "do_sample": do_sample,
        }
        if do_sample:
            generate_kwargs["temperature"] = temperature
            generate_kwargs["top_p"] = 1.0
        with torch.no_grad():
            output_ids = qwen.generate(**encoded, **generate_kwargs)
        new_ids = output_ids[0, prompt_length:]
        texts.append(qwen_tokenizer.decode(new_ids, skip_special_tokens=True))
    return texts


settings = [
    ("greedy", False, None),
    ("temperature 0.7", True, 0.7),
    ("temperature 1.5", True, 1.5),
]

for label, do_sample, temperature in settings:
    print("=" * 60)
    print(label)
    for i, text in enumerate(five_continuations(do_sample, temperature), start=1):
        print(f"\n[{i}] {text}")

greedy

[1] .  Given the context: What will Others want to do next?  Possible answers: go for

[2] .  Given the context: What will Others want to do next?  Possible answers: go for

[3] .  Given the context: What will Others want to do next?  Possible answers: go for

[4] .  Given the context: What will Others want to do next?  Possible answers: go for

[5] .  Given the context: What will Others want to do next?  Possible answers: go for
temperature 0.7

[1] , so what did people do?
Options:
- go to work
- have party
- go

[2] . It was a good day for hiking. In the morning, Mike went to hike. He started

[3]  so we decided to go for a hike in the park. Along the way, we came across a

[4] , so everyone decided to go for a picnic in the park. The total cost of renting the picnic

[5] , so my wife and I decided to go for a walk. We started our walk at 9
temperature 1.5

[1] , so what did people most likely to go out for? Choose the most suitable option to answer the

[2] . It was time to

### Difference between the three settings

Greedy decoding gave the same text all five times, a quiz line about what others will do next, because it always takes the most likely token. At temperature 0.7 the five continuations differ, but they stay with the sunny day: hiking, a picnic, and a walk. At temperature 1.5 they differ more sharply and drift off the prompt, into quiz instructions, a capitalization task, and broken phrasing such as "i planed" and "go outside and enjoy the day at home". Raising the temperature flattens the next-token distribution, so less likely tokens are chosen and the sentences hold together less well.

## 4. Key/value cache timing

Generate 200 new tokens, three times with `use_cache=True` and three times with `use_cache=False`. Decoding is greedy, so both settings produce the same tokens. The only change is whether past keys and values are kept.

One short generation runs first and is not timed. That loads the GPU kernels so the three measured runs are not paying for startup. `cuda.synchronize()` waits until the GPU is actually finished before the clock stops.

The runs with the cache off recompute the whole prefix at every new token. They can take several minutes.

In [ ]:
TIMING_PROMPT = "The capital of Turkey is"
TIMING_NEW_TOKENS = 200
N_RUNS = 3

timing_inputs = qwen_tokenizer(TIMING_PROMPT, return_tensors="pt").to(qwen.device)

with torch.no_grad():
    qwen.generate(
        **timing_inputs,
        max_new_tokens=1,
        do_sample=False,
        use_cache=True,
        pad_token_id=qwen_tokenizer.eos_token_id,
    )


def time_generation(use_cache):
    torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.no_grad():
        qwen.generate(
            **timing_inputs,
            max_new_tokens=TIMING_NEW_TOKENS,
            do_sample=False,
            use_cache=use_cache,
            pad_token_id=qwen_tokenizer.eos_token_id,
        )
    torch.cuda.synchronize()
    return time.perf_counter() - start


def mean_seconds(use_cache):
    runs = [time_generation(use_cache) for _ in range(N_RUNS)]
    print(f"use_cache={use_cache}: {runs}")
    return sum(runs) / len(runs)


kv_cache_seconds = {
    "with_cache": mean_seconds(True),
    "without_cache": mean_seconds(False),
}
kv_cache_seconds

### Why the cache is faster

With the cache on, the model keeps the key and value vectors it already computed for earlier tokens. Each new token only needs a forward pass for that one position, and the new vectors are appended. With the cache off, those vectors are discarded, so every new token recomputes keys and values for the entire prefix. The means above are higher without the cache because that repeated work grows as the text gets longer. The tokens do not change between the two settings, because both timings use greedy decoding.

# Part C — Three ways to classify

The dataset is `cornell-movie-review-data/rotten_tomatoes`. Each row is one movie-review sentence. Label 0 is negative and label 1 is positive. Fit or choose anything on `train` and `validation` only. The numbers you report are from `test`.

Three approaches, same test set:

1. A sentiment model used as-is, with no training.
2. Sentence embeddings plus a logistic regression trained on `train`.
3. The same embeddings, compared with two written label descriptions. The better wording is chosen on `validation`.

For each one, record accuracy, macro F1, the confusion matrix, and the seconds it takes to classify every test review. Training and the validation search stay outside that clock. If the GPU runs out of memory, restart the kernel and run Part C from this section. This part does not use the Qwen model.

## 1. Load the splits

`train` teaches the logistic regression. `validation` chooses the zero-shot wording. `test` is only for the final scores.

In [ ]:
import time

import numpy as np
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Uncomment if an import fails, then run this cell again.
# %pip install -q sentence-transformers scikit-learn

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
reviews = load_dataset("cornell-movie-review-data/rotten_tomatoes")

LABEL_NAMES = reviews["train"].features["label"].names


def split_xy(split):
    return list(split["text"]), list(split["label"])


train_texts, train_labels = split_xy(reviews["train"])
val_texts, val_labels = split_xy(reviews["validation"])
test_texts, test_labels = split_xy(reviews["test"])

print({name: len(reviews[name]) for name in reviews})
print("label names:", LABEL_NAMES)
print("example:", train_texts[0], "->", LABEL_NAMES[train_labels[0]])
print("device:", device)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/7.46k [00:00<?, ?B/s]

train.parquet: reconstructing file:   0%|          |  0.00B /  699kB            

train.parquet: downloading bytes:           |  0.00B            

validation.parquet: reconstructing file:   0%|          |  0.00B / 90.0kB            

...

test.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

{'train': 8530, 'validation': 1066, 'test': 1066}
label names: ['neg', 'pos']
example: the rock is destined to be the 21st century's new " conan " and that he's going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal . -> pos
device: cuda


In [ ]:
reviews["train"]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})

## 2. How each result is reported

Accuracy is the fraction of test reviews assigned the right label. Macro F1 averages the F1 of the negative class and the F1 of the positive class, so a model cannot look good by favoring one side. The confusion matrix rows are the true labels and the columns are the predicted labels, in the order given by `LABEL_NAMES`. The timer covers the whole test set and nothing else.

In [ ]:
def scores(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
    }


def timed(function):
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start = time.perf_counter()
    predictions = function()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    return predictions, time.perf_counter() - start


def report(name, y_true, y_pred, seconds):
    result = scores(y_true, y_pred)
    matrix = confusion_matrix(y_true, y_pred, labels=[0, 1])
    print(name)
    print(f"accuracy: {result['accuracy']:.4f}")
    print(f"macro F1: {result['macro_f1']:.4f}")
    print(f"seconds: {seconds:.3f}")
    print(f"rows are true {LABEL_NAMES}, columns are predicted {LABEL_NAMES}")
    print(matrix)
    print()
    return result


classification = {}

## 3. Task-specific model

`cardiffnlp/twitter-roberta-base-sentiment-latest` was trained to label tweets as negative, neutral, or positive. These reviews have no neutral label, and the model is not trained again.

Neutral is not folded into negative or positive. For each review the code compares the negative probability with the positive probability and keeps the larger one. If the model's top label is neutral, the review still receives whichever of the other two is more likely. That gives every test review a label the dataset can score.

In [ ]:
SENTIMENT_MODEL = "cardiffnlp/twitter-roberta-base-sentiment-latest"

sentiment_tokenizer = AutoTokenizer.from_pretrained(SENTIMENT_MODEL)
sentiment_model = AutoModelForSequenceClassification.from_pretrained(SENTIMENT_MODEL)
sentiment_model.to(device)
sentiment_model.eval()

id2label = {int(i): name.lower() for i, name in sentiment_model.config.id2label.items()}
print("model labels:", id2label)


def index_of(keyword):
    matches = [i for i, name in id2label.items() if keyword in name]
    if len(matches) != 1:
        raise RuntimeError(f"Expected one {keyword!r} label, found {matches} in {id2label}")
    return matches[0]


negative_index = index_of("negative")
positive_index = index_of("positive")


def predict_sentiment(texts, batch_size=32):
    predictions = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start : start + batch_size]
        encoded = sentiment_tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        ).to(device)
        with torch.no_grad():
            logits = sentiment_model(**encoded).logits
        probabilities = torch.softmax(logits, dim=-1)
        negative_vs_positive = torch.stack(
            [probabilities[:, negative_index], probabilities[:, positive_index]],
            dim=1,
        )
        predictions.append(negative_vs_positive.argmax(dim=1).cpu())
    return torch.cat(predictions).numpy()


sentiment_predictions, sentiment_seconds = timed(lambda: predict_sentiment(test_texts))
classification["task_specific"] = report(
    "task_specific",
    test_labels,
    sentiment_predictions,
    sentiment_seconds,
)

## 4. Embeddings plus logistic regression

`sentence-transformers/all-mpnet-base-v2` turns each review into one vector. A logistic regression learns, from the training vectors only, which direction in that space points toward a positive review. The clock starts after that fit, and it includes embedding the test reviews plus the classifier's `predict`.

In [38]:
EMBEDDING_MODEL = "sentence-transformers/all-mpnet-base-v2"
embedder = SentenceTransformer(EMBEDDING_MODEL, device=str(device))

train_embeddings = embedder.encode(
    train_texts,
    batch_size=64,
    convert_to_numpy=True,
    show_progress_bar=True,
)

classifier = LogisticRegression(max_iter=1000, random_state=SEED)
classifier.fit(train_embeddings, train_labels)


def predict_logreg():
    test_embeddings = embedder.encode(
        test_texts,
        batch_size=64,
        convert_to_numpy=True,
        show_progress_bar=False,
    )
    return classifier.predict(test_embeddings)


logreg_predictions, logreg_seconds = timed(predict_logreg)
classification["embeddings_logreg"] = report(
    "embeddings_logreg",
    test_labels,
    logreg_predictions,
    logreg_seconds,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/134 [00:00<?, ?it/s]

embeddings_logreg
accuracy: 0.8518
macro F1: 0.8518
seconds: 2.562
rows are true ['neg', 'pos'], columns are predicted ['neg', 'pos']
[[457  76]
 [ 82 451]]



## 5. Zero-shot with embeddings

No classifier is trained. Each review vector is compared, by cosine similarity, with a vector for a negative description and a vector for a positive description. The closer description wins.

Two wordings are scored on `validation`. The wording with the higher validation macro F1 is the one reported on `test`. The test set is not used to make that choice. You can edit the sentences below. Keep both wordings, and keep the choice on `validation`.

In [ ]:
PHRASINGS = {
    "short": {
        "negative": "This is a negative movie review.",
        "positive": "This is a positive movie review.",
    },
    "detailed": {
        "negative": "The reviewer disliked the film and thought it was bad.",
        "positive": "The reviewer enjoyed the film and thought it was good.",
    },
}


def zero_shot_predict(review_embeddings, descriptions):
    label_embeddings = embedder.encode(
        [descriptions["negative"], descriptions["positive"]],
        convert_to_numpy=True,
        show_progress_bar=False,
    )
    similarity = cosine_similarity(review_embeddings, label_embeddings)
    return similarity.argmax(axis=1)


val_embeddings = embedder.encode(
    val_texts,
    batch_size=64,
    convert_to_numpy=True,
    show_progress_bar=False,
)

best_name = None
best_f1 = -1.0
for name, descriptions in PHRASINGS.items():
    val_predictions = zero_shot_predict(val_embeddings, descriptions)
    val_result = scores(val_labels, val_predictions)
    print(
        f"{name}  validation accuracy={val_result['accuracy']:.4f}  "
        f"macro F1={val_result['macro_f1']:.4f}"
    )
    if val_result["macro_f1"] > best_f1:
        best_name = name
        best_f1 = val_result["macro_f1"]

print("chosen on validation:", best_name)
chosen_descriptions = PHRASINGS[best_name]


def predict_zero_shot():
    test_embeddings = embedder.encode(
        test_texts,
        batch_size=64,
        convert_to_numpy=True,
        show_progress_bar=False,
    )
    return zero_shot_predict(test_embeddings, chosen_descriptions)


zero_shot_predictions, zero_shot_seconds = timed(predict_zero_shot)
classification["zero_shot"] = report(
    "zero_shot",
    test_labels,
    zero_shot_predictions,
    zero_shot_seconds,
)

In [39]:
classification

{'task_specific': {'accuracy': 0.800187617260788,
  'macro_f1': 0.7989090725522547},
 'embeddings_logreg': {'accuracy': 0.851782363977486,
  'macro_f1': 0.8517776682624614},
 'zero_shot': {'accuracy': 0.7917448405253283, 'macro_f1': 0.7896651023890785}}

# Part D — Recommendation



## 1. Compare the three approaches and pick one

`recommended` starts as the approach with the highest test macro F1. Change that string if you would send the team a different one because it is faster or because it does not need labelled reviews. The string has to be exactly `task_specific`, `embeddings_logreg`, or `zero_shot`.

In [40]:
prediction_sets = {
    "task_specific": sentiment_predictions,
    "embeddings_logreg": logreg_predictions,
    "zero_shot": zero_shot_predictions,
}

print(f"{'approach':<20} {'accuracy':>10} {'macro_f1':>10}")
for name, result in classification.items():
    print(f"{name:<20} {result['accuracy']:10.4f} {result['macro_f1']:10.4f}")

recommended = max(classification, key=lambda name: classification[name]["macro_f1"])
print("recommended:", recommended)

approach               accuracy   macro_f1
task_specific            0.8002     0.7989
embeddings_logreg        0.8518     0.8518
zero_shot                0.7917     0.7897
recommended: embeddings_logreg


## 2. Five test reviews the recommended approach got wrong

In [41]:
recommended_predictions = prediction_sets[recommended]
mistakes = []
for text, gold, pred in zip(test_texts, test_labels, recommended_predictions):
    if int(gold) != int(pred):
        mistakes.append((text, LABEL_NAMES[int(gold)], LABEL_NAMES[int(pred)]))
    if len(mistakes) == 5:
        break

for i, (text, gold, pred) in enumerate(mistakes, start=1):
    print(f"[{i}] true={gold}  predicted={pred}")
    print(text)
    print()

[1] true=pos  predicted=neg
it's like a " big chill " reunion of the baader-meinhof gang , only these guys are more harmless pranksters than political activists .

[2] true=pos  predicted=neg
mostly , [goldbacher] just lets her complicated characters be unruly , confusing and , through it all , human .

[3] true=pos  predicted=neg
at its worst , the movie is pretty diverting ; the pity is that it rarely achieves its best .

[4] true=pos  predicted=neg
. . . a good film that must have baffled the folks in the marketing department .

[5] true=pos  predicted=neg
devotees of star trek ii : the wrath of khan will feel a nagging sense of deja vu , and the grandeur of the best next generation episodes is lacking .



## 3. What to tell the product team



I recommend the embeddings plus logistic regression. On the test set it reaches accuracy 0.852 and macro F1 0.852. The tweet sentiment model, used with no further training, reaches accuracy 0.800 and macro F1 0.799. Zero-shot comparison with the label descriptions chosen on validation reaches accuracy 0.792 and macro F1 0.790. The logistic regression is clearly the most accurate of the three.

That accuracy requires labelled data. It is fit on the 8,530 labelled training reviews. The tweet model needs no new labels, and zero-shot needs only the two description sentences. A gain of about five points of macro F1 over the tweet model is large enough that the team should use the labelled training set. If they had no labels at all, the tweet model would be the next choice, because it is slightly more accurate than zero-shot and is already trained.

A Turkish version cannot keep these English scores. In Part A, GPT-2 is the most expensive tokenizer for Turkish: fertility is 3.65 tokens per word in Turkish and 1.45 in English. Qwen is also more expensive for Turkish (2.69 versus 1.45). The Turkish BERT tokenizer is the cheap one, with Turkish fertility 1.78, and it is the only tokenizer for which Turkish uses fewer tokens per word than English. An English tokenizer spends more tokens on the same review, so less text fits in the context window and an API that bills per token costs more. The MPNet vectors and this logistic regression were learned from English, so the Turkish product also needs a Turkish embedding model, not only a cheaper tokenizer.

The five test reviews printed above are the ones this classifier misses. They share a split opinion inside one sentence: praise next to a complaint, or a negative judgment that still names something the reviewer liked. The embedding compresses both pulls into one vector, and the linear classifier follows the stronger pull to the wrong label.

## AI usage

Cursor's coding assistant was used to structure this notebook, explain the methods, and draft some of the written answers. The experiments were run in this notebook, and the numbers in the outputs and in `results.json` come from those runs. The assistant was also used to add this Part D section and the file writer above.